In [1]:
import torch
import tiktoken
from configs.model import ModelConfig
from configs.training import TrainingConfig
from configs.scheduler import SchedulerConfig
from configs.optimizer import OptimizerConfig
from configs.checkpoint import CheckpointConfig
from models.deepseek import DeepSeek
from generation.sample_text import generate, generate_sample_text, text_to_token_ids,token_ids_to_text
from datasets.preprocess import download_the_verdict,train_val_dataloader
from evaluation.losses import cross_entropy_loss,token_accuracy
from trainer.trainer import Trainer

c:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = tiktoken.get_encoding("gpt2")

In [3]:
DeepSeek_SMALL = ModelConfig(
    emb_dim=96,
    expert_dim=24,
    n_layers=12,
    n_heads=4,
    activation="gelu",
    context_length=24,
    latent_dim=48
)

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = DeepSeek(DeepSeek_SMALL).to(device)

In [5]:
text = "Every effort moves you "
tokenized_text = text_to_token_ids(text, tokenizer)
ids = generate(model, tokenized_text, max_new_tokens=20, context_size=DeepSeek_SMALL.context_length)
response = token_ids_to_text(ids, tokenizer)
print(response)

Every effort moves you  Oops


In [6]:
res_text = generate_sample_text(model,device='cpu',tokenizer=tokenizer,text=text,max_new_tokens=20)
print(res_text)

Every effort moves you  OopsrontalRail launchersParis pse sheddinghemy NationsMay Image allowed salient roomm arises herbs thriving Disaster exclusion TG


In [ ]:
TRAIN_CONFIG = TrainingConfig(
    epoch=5,
    batch_size=2,
    stride=24,
    context_length=24
)

OPTIMIZER_CONFIG = OptimizerConfig()
SCHEDULER_CONFIG = SchedulerConfig()
CHECKPOINT_CONFIG = CheckpointConfig()

In [8]:
raw_text = download_the_verdict()
train_dataloader,val_dataloader = train_val_dataloader(raw_text, TRAIN_CONFIG)

TrainingConfig(epoch=1, batch_size=2, stride=24, context_length=24, learning_rate=0.0003, weight_decay=0.1, grad_clip=1.0, mixed_precision=False, shuffle=False, num_workers=0, drop_last=True, gradient_accumulation_steps=1, train_data_ratio=0.9)
TrainingConfig(epoch=1, batch_size=2, stride=24, context_length=24, learning_rate=0.0003, weight_decay=0.1, grad_clip=1.0, mixed_precision=False, shuffle=False, num_workers=0, drop_last=True, gradient_accumulation_steps=1, train_data_ratio=0.9)


In [9]:
trainer=Trainer(model,tokenizer,train_dataloader,val_dataloader,device,cross_entropy_loss,token_accuracy,CHECKPOINT_CONFIG,TRAIN_CONFIG,OPTIMIZER_CONFIG,SCHEDULER_CONFIG)

In [10]:
trainer.fit()           ### use arg resume_latest=True or resume_best=True to resume training

100%|██████████| 96/96 [00:51<00:00,  1.88it/s]


Checkpoint saved -> checkpoints\checkpoint_96.pt
Best checkpoint saved -> checkpoints\best_checkpoint.pt
Output text:
 Every effort moves youacles
after 1 epoch global step 96 the train loss 10.467201481262842 val loss 9.364548683166504 and train acc| 0.017361111550902326 val acc| 0.03030303120613098 


### loss accuracy graph

In [ ]:
trainer.metrics.plot("train_loss","val_loss")

In [ ]:
trainer.metrics.plot("train_ppl","val_ppl")

In [11]:
trainer.metrics.metrics

defaultdict(list,
            {'train_loss': [10.467201481262842],
             'train_acc': [0.017361111550902326],
             'train_ppl': [35143.72923786341],
             'val_loss': [9.364548683166504],
             'val_acc': [0.03030303120613098],
             'val_ppl': [11667.33905194412]})